# Process Items JSONL

Reads raw metadata JSONL files, filters to only items that appear in `reviews.csv`, and writes `data/items.csv` for the downstream RecBole atomic-file preparation notebooks.

In [2]:
import json
import gzip
from pathlib import Path
from typing import Any
import pandas as pd

In [4]:
# --- Config ---
CATEGORIES = ["Beauty_and_Personal_Care", "Clothing_Shoes_and_Jewelry"]
FIELDS_TO_KEEP: list[str] = ["parent_asin", "title", "price", "store"]
DATA_DIR: str = "../data"
REVIEWS_FILE: str = "reviews.csv"
OUTPUT_FILE: str = "items.csv"

## Common utils

In [5]:
def stream_jsonl(path: str, fields: list[str] | None = None):
    if path.endswith(".gz"):
        with gzip.open(path, "rt", encoding="utf-8") as f:
            for _, line in enumerate(f):
                obj = json.loads(line)
                if fields is not None:
                    obj = {k: obj.get(k) for k in fields}
                yield obj
    else:
        with open(path, "r", encoding="utf-8") as f:
            for _, line in enumerate(f):
                obj = json.loads(line)
                if fields is not None:
                    obj = {k: obj.get(k) for k in fields}
                yield obj

## Load valid parent_asins from reviews

In [6]:
reviews_path = Path(DATA_DIR) / REVIEWS_FILE
reviews_df = pd.read_csv(reviews_path, usecols=["parent_asin"])
valid_asins: set[str] = set(reviews_df["parent_asin"].unique())
print(f"Loaded {len(valid_asins):,} unique parent_asins from {REVIEWS_FILE}")

Loaded 2,694,121 unique parent_asins from reviews.csv


## Stream and filter items from meta files

In [9]:
GZ_ZIPPED = False

items: list[dict[str, Any]] = []
for cat in CATEGORIES:
    path = f"{DATA_DIR}/meta_{cat}.jsonl" + (".gz" if GZ_ZIPPED else "")
    print(f"Loading items: {path}")
    for obj in stream_jsonl(path, fields=FIELDS_TO_KEEP):
        asin = obj.get("parent_asin")
        if asin is None or asin not in valid_asins:
            continue
        obj["category"] = cat
        items.append(obj)

print(f"Loaded {len(items):,} items")

Loading items: ../data/meta_Beauty_and_Personal_Care.jsonl
Loading items: ../data/meta_Clothing_Shoes_and_Jewelry.jsonl
Loaded 2,694,121 items


In [10]:
df = pd.DataFrame(items)
display(df.head())
df.info()

,parent_asin,title,price,store,category
0,B08BLDKYHB,"Shiyeen 10 Colors Hair Chalk for Girls Gift, K...",None,shiyeen,Beauty_and_Personal_Care
1,B0BM8WLSXF,"3 Inch Clipper Guards, Hair Clipper Guide Comb...",24.99,CR8GR8,Beauty_and_Personal_Care
2,B00N4LMZZK,Cathy Doll L-Glutathione Magic Cream SPF 50 Wh...,14.99,Cathy Doll,Beauty_and_Personal_Care
3,B01DX1OEFO,"L.A. COLORS 5 Color Matte Eyeshadow, Brown Twe...",2.49,L.A. COLORS,Beauty_and_Personal_Care
4,B08G1QHY8K,izneet 300PCS Eyelash Mascara Brushes Multicol...,None,Izneet,Beauty_and_Personal_Care


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2694121 entries, 0 to 2694120
Data columns (total 5 columns):
 #   Column       Dtype 
---  ------       ----- 
 0   parent_asin  object
 1   title        object
 2   price        object
 3   store        object
 4   category     object
dtypes: object(5)
memory usage: 102.8+ MB


In [11]:
# Clean price: Amazon metadata sometimes uses "from X.XX" format.
# Extract the first numeric value to keep only the number.
df["price"] = (
    df["price"]
    .astype(str)
    .str.extract(r"(\d+\.?\d*)", expand=False)
    .replace("", None)
)

## Export to CSV

In [12]:
output_path = Path(DATA_DIR)
output_path.mkdir(parents=True, exist_ok=True)
file_path = output_path / OUTPUT_FILE
df.to_csv(file_path, index=False)
print(f"Wrote {file_path} ({df.shape[0]:,} rows, {df.shape[1]} columns)")

Wrote ../data/items.csv (2,694,121 rows, 5 columns)
